# Day 7 — Data Visualization & SQL
**Date:** 16-Jul-2026

Today's topics:
- Matplotlib basics
- Basic charts (line, bar)
- Histograms
- Scatter plots
- Correlation heatmaps (concept)
- SQL basics — `SELECT`, `WHERE`, `GROUP BY`, `JOIN`
- Connecting Python with databases

This notebook is self-contained: it generates its own sample data and its own SQLite database in-memory/on-disk, so every section runs top-to-bottom without external files.


## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

np.random.seed(7)
print("Ready.")


## 2. Sample Dataset

We'll simulate a store's daily sales data — the same kind of dataset a real analyst would visualize.

In [ ]:
n_days = 90
dates = pd.date_range("2026-04-01", periods=n_days, freq="D")

categories = ["Electronics", "Grocery", "Clothing", "Books"]
category_choice = np.random.choice(categories, size=n_days)

# Correlated-ish features: more footfall -> more revenue, with noise
footfall = np.random.poisson(lam=200, size=n_days) + np.arange(n_days) // 3
ad_spend = np.round(np.random.uniform(50, 500, size=n_days), 2)
revenue = np.round(footfall * np.random.uniform(3, 6) + ad_spend * 1.5 + np.random.normal(0, 150, n_days), 2)
units_sold = np.round(revenue / np.random.uniform(15, 25, size=n_days)).astype(int)

df = pd.DataFrame({
    "date": dates,
    "category": category_choice,
    "footfall": footfall,
    "ad_spend": ad_spend,
    "units_sold": units_sold,
    "revenue": revenue,
})

df.head()


## 3. Matplotlib Basics

The core pattern: create a `Figure`/`Axes`, plot onto it, then label and show.

In [ ]:
fig, ax = plt.subplots()

daily_revenue = df.groupby("date")["revenue"].sum()
ax.plot(daily_revenue.index, daily_revenue.values, color="#2F5496", linewidth=1.5)

ax.set_title("Daily Revenue — Apr to Jun 2026")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (₹)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 4. Basic Charts — Bar Chart

In [ ]:
revenue_by_category = df.groupby("category")["revenue"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
bars = ax.bar(revenue_by_category.index, revenue_by_category.values,
              color=["#2F5496", "#8FAADC", "#C6D9F1", "#1F3864"])

ax.set_title("Total Revenue by Category")
ax.set_xlabel("Category")
ax.set_ylabel("Revenue (₹)")

# Annotate bars with their values
for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height:,.0f}", (bar.get_x() + bar.get_width()/2, height),
                textcoords="offset points", xytext=(0, 4), ha="center", fontsize=9)

plt.tight_layout()
plt.show()


## 5. Histograms

Histograms show the *distribution* of a single numeric variable — here, daily revenue.

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["revenue"], bins=15, color="#4472C4", edgecolor="white")

ax.set_title("Distribution of Daily Revenue")
ax.set_xlabel("Revenue (₹)")
ax.set_ylabel("Frequency (days)")

plt.tight_layout()
plt.show()


## 6. Scatter Plots

Scatter plots reveal the *relationship* between two numeric variables — here, footfall vs. revenue.

In [ ]:
fig, ax = plt.subplots()
scatter = ax.scatter(df["footfall"], df["revenue"], c=df["ad_spend"], cmap="viridis", alpha=0.75)

ax.set_title("Footfall vs. Revenue (colored by Ad Spend)")
ax.set_xlabel("Footfall")
ax.set_ylabel("Revenue (₹)")

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Ad Spend (₹)")

plt.tight_layout()
plt.show()


## 7. Correlation Heatmaps (Concept)

A correlation heatmap shows how every numeric column relates to every other one, on a scale from
**-1** (perfect negative relationship) to **+1** (perfect positive relationship). We compute the
correlation matrix with pandas, then render it manually with `imshow` (no seaborn needed —
matplotlib alone is enough for the concept).

In [ ]:
numeric_cols = ["footfall", "ad_spend", "units_sold", "revenue"]
corr = df[numeric_cols].corr()
corr


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha="right")
ax.set_yticklabels(numeric_cols)

# Annotate each cell with its correlation value
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black", fontsize=9)

ax.set_title("Correlation Heatmap")
fig.colorbar(im, ax=ax, label="Correlation coefficient")
plt.tight_layout()
plt.show()


## 8. SQL Basics — Connecting Python with a Database

We'll spin up a local SQLite database (built into Python, no server needed), load our
sales DataFrame plus a small `stores` lookup table into it, and then practice SQL:
`SELECT`, `WHERE`, `GROUP BY`, and `JOIN`.

In [ ]:
# Connect (creates the file if it doesn't exist)
conn = sqlite3.connect("day7_sales.db")

# Load the sales DataFrame into a table
df_sql = df.copy()
df_sql["date"] = df_sql["date"].astype(str)
df_sql.to_sql("sales", conn, if_exists="replace", index=False)

# A small lookup table to demonstrate JOIN
stores = pd.DataFrame({
    "category": ["Electronics", "Grocery", "Clothing", "Books"],
    "department_head": ["Rahul", "Meera", "Ananya", "Sanjay"],
    "floor": [2, 1, 3, 1],
})
stores.to_sql("stores", conn, if_exists="replace", index=False)

print("Tables loaded:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn))


### 8.1 `SELECT` — pick specific columns

In [ ]:
query = """
SELECT date, category, revenue
FROM sales
LIMIT 5;
"""
pd.read_sql(query, conn)


### 8.2 `WHERE` — filter rows

In [ ]:
query = """
SELECT date, category, revenue, footfall
FROM sales
WHERE category = 'Electronics' AND revenue > 1000
ORDER BY revenue DESC
LIMIT 5;
"""
pd.read_sql(query, conn)


### 8.3 `GROUP BY` — aggregate per category

In [ ]:
query = """
SELECT category,
       COUNT(*)        AS days_count,
       ROUND(AVG(revenue), 2)   AS avg_revenue,
       ROUND(SUM(revenue), 2)   AS total_revenue
FROM sales
GROUP BY category
ORDER BY total_revenue DESC;
"""
pd.read_sql(query, conn)


### 8.4 `JOIN` — combine `sales` with `stores`

In [ ]:
query = """
SELECT s.category,
       st.department_head,
       st.floor,
       ROUND(SUM(s.revenue), 2) AS total_revenue
FROM sales AS s
JOIN stores AS st
  ON s.category = st.category
GROUP BY s.category, st.department_head, st.floor
ORDER BY total_revenue DESC;
"""
pd.read_sql(query, conn)


In [ ]:
conn.close()
print("Connection closed.")


---
### Summary — what we covered today
1. **Matplotlib basics** — Figure/Axes pattern, titles, labels
2. **Basic charts** — line chart (revenue over time), bar chart (revenue by category)
3. **Histograms** — distribution of daily revenue
4. **Scatter plots** — relationship between footfall and revenue, colored by a third variable
5. **Correlation heatmaps (concept)** — computing and visualizing a correlation matrix
6. **SQL basics** — `SELECT`, `WHERE`, `GROUP BY`, `JOIN` against a local SQLite database
7. **Connecting Python with databases** — `sqlite3` + `pandas.read_sql` / `to_sql`

**Next step ideas:** try `seaborn` for statistical plots with less boilerplate, and practice SQL
subqueries and window functions (`RANK()`, `ROW_NUMBER()`) on the same `sales` table.
